[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs/Lab_5_NonHomogeneous_PDEs.ipynb) [![Launch JupyterLab](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hany73-max/PDE-Rover-Engine/main?urlpath=lab/tree/labs/Lab_5_NonHomogeneous_PDEs.ipynb)


In [ ]:
# 🚀 GOOGLE COLAB SETUP
# If you are running this notebook on Google Colab, execute this cell to clone the repo and set paths:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] Google Colab setup complete! Connected to PDE-Rover-Engine repository.')
except NameError:
    pass


# 🔋 Lab 5: Non-Homogeneous Linear PDEs, Reducible Operators, and Narrow-Corridor Decay
### *Mission Control: Why Dirichlet Obstacles (`dirichlet_potential`) Fail in Narrow Passages*
**Course:** Applied Partial Differential Equations & Planetary Robotics
**Lecture Notes Reference:** Chapter 2, Section 2.1 — **Notes Chapter 2 pp. 1–10** *(PDF pp. 52–62; Reducible Linear Factor Rule on Ch. 2 p. 2 / PDF p. 54, Worked Ex. 5 on Ch. 2 p. 4 / PDF p. 56)*

---
### 🗺️ Where You Are in the Rover Navigation Pipeline
```
[Labs 1–3: Slopes, Streamlines, & Safety Buffers] ──► [Lab 4: Laplace Theory] ──► 📍 [Lab 5: Reducible Operators & Corridor Decay] ──► [Lab 6: Complete Rover Planner (SOR + Neumann)]
```

---
### 🧭 The Rosetta Stone: Connecting Lecture Notes to the Rover Engine

| 1. Lecture Notes Math (PDF pp. 52–62) | 2. Geometric & Physical Meaning | 3. Real Project Function (`rover_path_finder/`) |
| :--- | :--- | :--- |
| $F(D_x, D_y)z = 0$ (non-homogeneous in order) | Operator mixing derivatives of different orders or a constant term $c$ (e.g. $aD_x + bD_y + c$) | Wave propagation coupled with exponential growth/decay |
| $(a D_x + b D_y + c)z = 0 \implies z = e^{-\frac{c}{a}x}\phi(ay - bx)$ | Every constant offset $c \neq 0$ introduces an exponential envelope $e^{-(c/a)x}$ | Exponential attenuation along a channel |
| $(F_1 \cdot F_2)z = 0 \implies z = z_1 + z_2$ | Factoring a reducible operator into linear factors and summing solutions | Superposition of decaying modes |
| Corridor gradient decay $\|\nabla\phi\| \propto e^{-\pi d / w}$ | Fixing canyon walls at $\phi=1$ causes the gradient to decay exponentially inside narrow passages of width $w$ | Why [`planners.neumann_potential`](rover_path_finder/planners.py) replaces [`planners.dirichlet_potential`](rover_path_finder/planners.py) |

---
## 1. Terminology Clarification: What Does "Non-Homogeneous" Mean in Chapter 2 of the Notes?

When you open **Chapter 2 of the lecture notes (PDF p. 52): "Non-Homogeneous Linear PDEs"**, notice a crucial distinction in terminology:
* In introductory ODEs, "non-homogeneous" usually means a non-zero right-hand side $f(x, y) \neq 0$.
* In **Chapter 2 of our PDE notes (PDF pp. 52–57)**, **"Non-Homogeneous" means the differential operator $F(D_x, D_y)$ on the left-hand side contains derivatives of different orders** (such as a 1st derivative $D_x$ mixed with a 0th-order constant term $c$), even when the right-hand side is zero:
  $$(2 D_x - 3 D_y + 1)(D_x + 2 D_y - 2)z = 0$$

---
## 2. How Does Exponential Decay Connect to `rover_path_finder/planners.py`?

In [`rover_path_finder/planners.py`](rover_path_finder/planners.py), two harmonic potential planners are implemented:
1. **Formulation A (`dirichlet_potential`):** Fixes all obstacle walls at a constant high potential $\phi = 1$ and the goal at $\phi = 0$.
2. **Formulation B (`neumann_potential`):** Treats obstacle walls as insulated zero-flux boundaries ($\partial\phi/\partial n = 0$) with a point source at `start` ($\phi=1$) and point sink at `goal` ($\phi=0$).

**Why does `dirichlet_potential` fail when the rover must pass through a narrow canyon of width $w$?**
Inside a narrow passage of width $w$ whose side walls are clamped at $\phi = 1$, the transverse profile $\sin(\pi y / w)$ introduces an effective constant term $c = \frac{\pi}{w}$ along the corridor axis $x$:
$$\left(D_x + \frac{\pi}{w}\right)\tilde{\phi} = 0 \implies \tilde{\phi}(x, y) \propto e^{-\frac{\pi}{w} x} \sin\left(\frac{\pi y}{w}\right)$$
Because of the exponential factor $e^{-\pi x / w}$, the potential flattens to $\phi \approx 1.000000$ within a few pixels of entering the canyon, causing floating-point underflow and stalling the rover!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab5

%matplotlib inline
print("Lab 5 Environment Initialized! Ready for Reducible Operators & Corridor Decay Analysis.")


---
## 3. Complete Chapter 2 Recipe: Cases (A), (B), (C), (D) & Particular Integrals (PDF pp. 52–62)

### Case (A): Single or Repeated Linear Factor $(a D_x + b D_y + c)^k z = 0$ *(Ch. 2 pp. 2–3 / PDF pp. 54–55)*

#### 1. Where does $z = e^{-\frac{c}{a}x}\phi(a y - b x)$ come from?
1. Rewrite $(a D_x + b D_y + c)z = 0$ as $a p + b q = -c z$ (a Lagrange PDE from Lab 2!).
2. Lagrange's auxiliary equations are:
   $$\frac{dx}{a} = \frac{dy}{b} = \frac{dz}{-c z}$$
3. From the first two fractions: $b\,dx - a\,dy = 0 \implies a y - b x = c_1$.
4. From the first and third fractions ($a \neq 0$): $\frac{dz}{z} = -\frac{c}{a}dx \implies z = c_2 e^{-\frac{c}{a}x}$.
5. Combining both invariants gives the **Linear Factor Theorem**:
   * **When $a \neq 0$:**
     $$\boxed{(a D_x + b D_y + c)z = 0 \implies z = e^{-\frac{c}{a}x}\,\phi(a y - b x)}$$
   * **When $a = 0$ and $b \neq 0$ (or pairing $\frac{dy}{b} = \frac{dz}{-cz}$):**
     $$\boxed{z = e^{-\frac{c}{b}y}\,\phi(a y - b x)}$$

#### 2. What if a linear factor is repeated $k$ times? $(a D_x + b D_y + c)^k z = 0$ (Ch. 2 pp. 2–3 / PDF pp. 54–55)
* **Using the $x$-exponential ($a \neq 0$):** Multiply successive arbitrary functions by $1, x, x^2, \dots, x^{k-1}$:
  $$\boxed{z = e^{-\frac{c}{a}x}\left[\phi_1(ay - bx) + x\,\phi_2(ay - bx) + x^2\,\phi_3(ay - bx) + \dots + x^{k-1}\phi_k(ay - bx)\right]}$$
* **Using the $y$-exponential ($b \neq 0$, especially when $a = 0$):** Multiply by $1, y, y^2, \dots, y^{k-1}$:
  $$\boxed{z = e^{-\frac{c}{b}y}\left[\phi_1(ay - bx) + y\,\phi_2(ay - bx) + \dots + y^{k-1}\phi_k(ay - bx)\right]}$$

> 📌 **PDF Pages 54–55 Typo Alert (Chapter 2 pp. 2–3, Examples 3 & 4):**
> * In **Example 3 (PDF p. 54):** $(2D_x - 3D_y + 2)^2 z = 0$ ($a=2, b=-3, c=2$), the first formula $z = e^{-x}[\phi_1(2y+3x) + x\phi_2(2y+3x)]$ is **100% correct**, whereas the alternative line ("Or $z = e^{\frac{2}{3}y}[\dots]$") mistakenly wrote $x\phi_2$ instead of $y\phi_2$.
> * In **Example 4 (PDF p. 55):** $(D_x - 2D_y + 1)^4 z = 0$ ($a=1, b=-2, c=1, k=4$), the bracket uses powers of $x$ ($1, x, x^2, x^3$), so the prefactor outside the bracket must be $\boxed{e^{-(c/a)x} = e^{-x}}$ (the PDF accidentally printed $e^{\frac{1}{2}y}$ outside the $x$-bracket).
> * **Exam Rule of Thumb:** Whenever $a \neq 0$, always use the standard $x$-form $e^{-(c/a)x}[\phi_1 + x\phi_2 + \dots]$!

---
### Case (B): Reducible Product of Distinct Linear Factors *(Ch. 2 pp. 3–4 / PDF pp. 55–56)*
When $F(D_x, D_y)$ factors into linear operators:
$$(a_1 D_x + b_1 D_y + c_1)(a_2 D_x + b_2 D_y + c_2)z = 0$$
by superposition, the general solution is the sum of the solutions to each factor:
$$z = e^{-\frac{c_1}{a_1}x}\Phi_1(a_1 y - b_1 x) + e^{-\frac{c_2}{a_2}x}\Phi_2(a_2 y - b_2 x)$$

#### 📖 Worked Example (Chapter 2 p. 4, Example 5 / PDF p. 56)
**Problem:** Solve:
$$(2 D_x - 3 D_y + 1)(D_x + 2 D_y - 2)z = 0$$
1. **First factor $(2 D_x - 3 D_y + 1)z = 0$:**
   * $a_1 = 2,\; b_1 = -3,\; c_1 = 1 \implies -\frac{c_1}{a_1}x = -\frac{1}{2}x$ and $a_1 y - b_1 x = 2y - (-3)x = 2y + 3x$.
   * Solution 1: $z_1 = e^{-x/2}\Phi_1(2y + 3x)$.
2. **Second factor $(D_x + 2 D_y - 2)z = 0$:**
   * $a_2 = 1,\; b_2 = 2,\; c_2 = -2 \implies -\frac{c_2}{a_2}x = 2x$ and $a_2 y - b_2 x = y - 2x$.
   * Solution 2: $z_2 = e^{2x}\Phi_2(y - 2x)$.
3. **General Solution:** $\boxed{z = e^{-x/2}\Phi_1(2y + 3x) + e^{2x}\Phi_2(y - 2x)}$.

*(Similarly, in **Example 7, Ch. 2 p. 5 / PDF p. 57**, $(D_x^2 - D_y^2 - 3D_x + 3D_y)z = 0$ factors by grouping as $(D_x - D_y)(D_x + D_y - 3)z = 0 \implies \boxed{z = \phi_1(y + x) + e^{3x}\phi_2(y - x)}$!)*

---
### Cases (C) & (D): Irreducible Operators *(Ch. 2 pp. 5–8 / PDF pp. 57–60)*
* **When to use Case (C):** When $F(D_x, D_y)$ cannot be factored into linear 1st-order terms—for example, $(D_x - D_y^3)z = 0$, $(D_x^2 + D_x + D_y)z = 0$, or $(D_x^2 - D_y)z = 0$.
* **Lecture Notes Rule (Ch. 2 pp. 5–6 / PDF pp. 57–58):**
  Assume an infinite exponential series trial solution:
  $$\boxed{z = \sum_{i=1}^{\infty} A_i\, e^{a_i x + b_i y} \qquad \text{subject to the characteristic relation } F(a_i, b_i) = 0}$$
* **📖 Worked Examples 8, 9, & 11 (Ch. 2 pp. 6–7 / PDF pp. 58–59):**
  1. **Example 8:** Solve $(D_x - D_y^3)z = 0$.
     * Replacing $D_x \to a_i,\; D_y \to b_i$ gives $a_i - b_i^3 = 0 \implies a_i = b_i^3$.
     * General solution: $\boxed{z = \sum_{i=1}^{\infty} A_i\, e^{b_i^3 x + b_i y}}$.
  2. **Example 9:** Solve $(D_x^2 + D_x + D_y)z = 0$.
     * Relation: $a_i^2 + a_i + b_i = 0 \implies b_i = -a_i(a_i + 1) \implies \boxed{z = \sum_{i=1}^{\infty} A_i\, e^{a_i x - a_i(a_i + 1)y}}$.
  3. **Case (D) — Combined Reducible & Irreducible Factors (Example 11, Ch. 2 p. 7 / PDF p. 59):**
     * For $(D_x - D_y - 1)(D_x - D_y^2)z = 0$: add the reducible C.F. $e^x\phi(y+x)$ and the irreducible C.F. $\sum_{i=1}^\infty A_i e^{b_i^2 x + b_i y}$:
       $$\boxed{z = e^x \phi(y + x) + \sum_{i=1}^{\infty} A_i e^{b_i^2 x + b_i y}}$$

---
### Part D: Particular Integrals in Chapter 2 & The Conjugate Operator Trick (Ch. 2 pp. 8–10 / PDF pp. 60–62)

To find $z_2 = \frac{1}{F(D_x, D_y)}f(x, y)$ when $F(D_x, D_y)$ is non-homogeneous in order, the lecture notes highlight **two essential techniques**:

#### 1. Polynomial Expansion by Factoring Out the Lowest Derivative (Example 12, Ch. 2 p. 8 / PDF p. 60)
* **Problem:** Find the Particular Integral of $(D_x^2 - D_y^2 + D_x)z = x^2 + 2y$.
  * Factor out the **lowest-order term $D_x$** from the denominator:
    $$z_2 = \frac{1}{D_x\left[1 - \left(\frac{D_y^2}{D_x} - D_x\right)\right]}(x^2 + 2y) = \frac{1}{D_x}\left[1 + \left(\frac{D_y^2}{D_x} - D_x\right) + \left(\frac{D_y^2}{D_x} - D_x\right)^2 + \dots\right](x^2 + 2y)$$
  * Since $D_y^2(x^2 + 2y) = 0$, all $D_y^2$ terms vanish, leaving:
    $$z_2 = \frac{1}{D_x}\left(1 - D_x + D_x^2\right)(x^2 + 2y) = \frac{1}{D_x}\left[(x^2 + 2y) - 2x + 2\right] = \boxed{\frac{x^3}{3} + 2xy - x^2 + 2x}$$

#### 2. The Conjugate Operator Trick for $\sin/\cos$ When a 1st Derivative Remains! (Example 14, Ch. 2 pp. 9–10 / PDF pp. 61–62)
* **Problem:** Solve $(D_x^2 - D_y)z = \sin(x - 2y)$.
  1. **Complementary Function $z_1$:** Since $a_i^2 - b_i = 0 \implies b_i = a_i^2$, $z_1 = \sum_{i=1}^\infty A_i e^{a_i x + a_i^2 y}$.
  2. **Particular Integral $z_2$:** For $\sin(x - 2y)$, $a = 1$ and $b = -2$.
     * The sine shortcut only replaces **2nd-order derivatives** ($D_x^2 \to -a^2 = -1$, $D_y^2 \to -b^2 = -4$), leaving a 1st-order $D_y$ in the denominator:
       $$z_2 = \frac{1}{D_x^2 - D_y}\sin(x - 2y) = \frac{1}{-1 - D_y}\sin(x - 2y)$$
     * **The Conjugate Trick:** Multiply top and bottom by the **conjugate operator $(-1 + D_y)$** to turn the denominator into $1 - D_y^2$:
       $$z_2 = \frac{-1 + D_y}{(-1 - D_y)(-1 + D_y)}\sin(x - 2y) = \frac{-1 + D_y}{1 - D_y^2}\sin(x - 2y)$$
     * Now substitute $D_y^2 \to -b^2 = -(-2)^2 = -4$ in the denominator:
       $$z_2 = \frac{-1 + D_y}{1 - (-4)}\sin(x - 2y) = \frac{1}{5}(-1 + D_y)\sin(x - 2y)$$
     * Finally, apply $(-1 + D_y)$ to $\sin(x - 2y)$ (where $D_y[\sin(x - 2y)] = -2\cos(x - 2y)$):
       $$\boxed{z_2 = \frac{1}{5}\big[-\sin(x - 2y) - 2\cos(x - 2y)\big]}$$


---
## 4. 📝 Paper & Pen Challenge 1 (Chapter 2 p. 4, Example 5 / PDF p. 56)

**Problem:** Find the general solution of the reducible PDE:
$$(2 D_x - 3 D_y + 1)(D_x + 2 D_y - 2)z = 0$$

Write the general solution using `exp(...)`, `Phi1(...)`, and `Phi2(...)`:

<details>
<summary>💡 <b>Hint 1 (First Factor)</b></summary>

For $(2 D_x - 3 D_y + 1)$, $a_1=2, b_1=-3, c_1=1 \implies \text{exp}(-x/2)\text{*Phi1}(2y + 3x)$.
</details>

<details>
<summary>💡 <b>Hint 2 (Second Factor & Sum)</b></summary>

For $(D_x + 2 D_y - 2)$, $a_2=1, b_2=2, c_2=-2 \implies \text{exp}(2x)\text{*Phi2}(y - 2x)$. Add both terms together.
</details>


In [ ]:
# GRADED CELL: Paper Challenge 1
### START CODE HERE ###
student_solution = "exp(-x/2)*Phi1(2y + 3x) + exp(2x)*Phi2(y - 2x)"
### END CODE HERE ###

# Run numerical verification test:
test_lab5.check_paper_challenge_1(student_solution)


---
## 5. 🚀 Paper & Pen Challenge 2 (Computing the Exponential Damping Rate $k = -c/a$)

**Problem:** Given the linear operator equation:
$$(D_x + 2 D_y + 1)z = 0$$

**On your paper:**
1. Identify the coefficients $a, b, c$ from $(a D_x + b D_y + c)z = 0$.
2. Compute the exponential rate $k = -\frac{c}{a}$ in $z = e^{k x}\Phi(a y - b x)$.

<details>
<summary>💡 <b>Hint (Extracting a and c)</b></summary>

Here $a = 1$ (coefficient of $D_x$) and $c = 1$ (constant term), so $k = -c/a = -1/1$.
</details>


In [ ]:
# GRADED CELL: Exponential Damping Rate
### START CODE HERE ###
damping_rate = -1.0
### END CODE HERE ###

# Run numerical verification test:
test_lab5.check_paper_challenge_2(damping_rate)


---
## 6. 💻 Python Implementation: Linear Factor Coefficient Parser

Let's write a helper `parse_linear_factor(factor_str)` that extracts the numerical tuple `(a, b, c)` from a linear factor string like `"2*D_x - 3*D_y - 5"`.


In [ ]:
# GRADED FUNCTION: parse_linear_factor
def parse_linear_factor(factor_str: str) -> tuple[float, float, float]:
    # Parses coefficients a, b, c from a linear factor string "a*D_x + b*D_y + c"
    ### START CODE HERE ###
    import re
    ma = re.search(r'([+-]?\s*\d*)\s*\*?\s*D_x', factor_str)
    mb = re.search(r'([+-]?\s*\d*)\s*\*?\s*D_y', factor_str)
    mc = re.search(r'([+-]?\s*\d+)\s*$', factor_str.replace(" ", ""))

    def to_float(val, default):
        if not val or val.strip() in ['+', '']: return default
        if val.strip() == '-': return -default
        return float(val.replace(" ", ""))

    a = to_float(ma.group(1), 1.0) if ma else 0.0
    b = to_float(mb.group(1), 1.0) if mb else 0.0
    c = float(mc.group(1)) if mc else 0.0
    ### END CODE HERE ###
    return a, b, c

# Run automated unit test:
test_lab5.test_linear_factor_evaluator(parse_linear_factor)


---
## 7. 🎨 Visualizing the Physics & Connecting to `rover_path_finder/planners.py`

Run the cell below to see two plots:
1. **Left:** The exponentially damped wave $z = e^{-0.5x}\sin(y - 2x)$ satisfying $(2D_x + D_y + 1)z = 0$.
2. **Right:** A direct comparison using [`planners.dirichlet_potential`](rover_path_finder/planners.py) vs. [`planners.neumann_potential`](rover_path_finder/planners.py) through a narrow 8-pixel canyon, demonstrating why Dirichlet walls suffer exponential gradient decay $e^{-\pi x / w}$ while Neumann walls maintain a smooth linear gradient!


In [ ]:
from rover_path_finder.planners import dirichlet_potential, neumann_potential

x = np.linspace(0, 5, 50)
y = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x, y)
Z = np.exp(-0.5 * X) * np.sin(Y - 2.0 * X)

corridor_hazard = np.zeros((40, 60), dtype=bool)
corridor_hazard[:16, 15:45] = True   # Upper canyon wall
corridor_hazard[24:, 15:45] = True   # Lower canyon wall (corridor width w = 8 pixels)
start_pt, goal_pt = (8, 20), (52, 20)

phi_dir, _, _ = dirichlet_potential(corridor_hazard, goal=goal_pt, tol=1e-8)
phi_neu, _, _ = neumann_potential(corridor_hazard, start=start_pt, goal=goal_pt, tol=1e-6)

fig = plt.figure(figsize=(14, 5.5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='plasma', alpha=0.88)
ax1.set_title(r"1. Linear Factor Solution: $z = e^{-0.5x}\sin(y - 2x)$", fontsize=11)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel("z")
plt.colorbar(surf, ax=ax1, shrink=0.55)

ax2 = fig.add_subplot(1, 2, 2)
center_x = np.arange(8, 53)
ax2.plot(center_x, 1.0 - phi_dir[20, 8:53], 'r--', linewidth=2.5, label=r"1 - dirichlet_potential (Exponential Decay $e^{-\pi x/w}$)")
ax2.plot(center_x, phi_neu[20, 8:53], 'b-', linewidth=2.5, label="neumann_potential (Smooth Linear Gradient)")
ax2.set_yscale('log')
ax2.set_title("2. Real Engine: Narrow Corridor Gradient Decay (Log Scale)", fontsize=11)
ax2.set_xlabel("Position along narrow corridor (x)")
ax2.set_ylabel("Potential Drop (log scale)")
ax2.grid(True, which="both", alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print("Lab 5 Takeaways:")
print("  1) A linear factor (a*D_x + b*D_y + c)z = 0 introduces an exponential envelope exp(-(c/a)*x).")
print("  2) In narrow corridors of width w, Dirichlet walls cause exponential gradient decay exp(-pi*x/w).")
print("  3) Switching to insulated Neumann walls (neumann_potential) eliminates corridor decay!")
